In [1]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [2]:
# loading the data from the pivoted csv
water_df = pd.read_csv("water_pivoted.csv")

In [3]:
# dropping any rows that are missing the target variable of total coliforms
water_df = water_df.dropna(subset = ['Total Coliforms'])

In [4]:
# dropping variables such as sample site and date which will not be included in analysis
water_df = water_df.drop(columns=['Sample Site', 'Latitude (Degrees)', 'Longitude (Degrees)', 'Sample Date', 'Site Key'])

In [5]:
# creates a binary column where contaminated water is a 1 and safe water is a 0
# total coliforms > 235 are considered contaminated, total coliforms < 235 are considered safe
water_df["contaminated"] = (water_df["Total Coliforms"] > 235).astype(int)
water_df.head()

,(Aminomethyl)phosphonic acid,"2,4-D","2,4-DB","2,4-DP","2,4-Dichlorophenol",3-(Methylphosphinico)propionic acid,4-Chloro-2-methylphenol,Aldicarb,Aldicarb sulfone,Aldicarb sulfoxide,...,Vinclozolin,Water Temperature (Field),Zinc (Zn)(Extractable),Zinc (Zn)(Total),alpha-BHC,alpha-Endosulfan,gamma-BHC,"p,p-Methoxychlor",pH,contaminated
101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16.75,NaN,NaN,NaN,NaN,NaN,NaN,8.4,0
102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16.28,NaN,NaN,NaN,NaN,NaN,NaN,8.4,0
103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,14.06,NaN,NaN,NaN,NaN,NaN,NaN,8.3,0
104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,12.10,NaN,NaN,NaN,NaN,NaN,NaN,8.2,0
105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,8.12,NaN,NaN,NaN,NaN,NaN,NaN,8.2,0


In [6]:
# checks for class balance
# the classes of contaminated versus safe water are fairly well balanced
water_df.contaminated.value_counts()

contaminated
0    5333
1    4751
Name: count, dtype: int64

In [7]:
# E.coli is a coliform and is included in total coliforms as a result we will not include it as a predictor
water_df = water_df.drop(columns=['E.coli'])
water_df.head()

,(Aminomethyl)phosphonic acid,"2,4-D","2,4-DB","2,4-DP","2,4-Dichlorophenol",3-(Methylphosphinico)propionic acid,4-Chloro-2-methylphenol,Aldicarb,Aldicarb sulfone,Aldicarb sulfoxide,...,Vinclozolin,Water Temperature (Field),Zinc (Zn)(Extractable),Zinc (Zn)(Total),alpha-BHC,alpha-Endosulfan,gamma-BHC,"p,p-Methoxychlor",pH,contaminated
101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16.75,NaN,NaN,NaN,NaN,NaN,NaN,8.4,0
102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16.28,NaN,NaN,NaN,NaN,NaN,NaN,8.4,0
103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,14.06,NaN,NaN,NaN,NaN,NaN,NaN,8.3,0
104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,12.10,NaN,NaN,NaN,NaN,NaN,NaN,8.2,0
105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,8.12,NaN,NaN,NaN,NaN,NaN,NaN,8.2,0


In [8]:
# drops total coliforms from dataset, contaminated column will be used as target variable
water_df = water_df.drop(columns=['Total Coliforms'])

In [9]:
# splits dataset into target variable and predictors
y = water_df['contaminated']
X = water_df.drop('contaminated', axis = 1)

In [10]:
# splits data into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

There are many columns with large portions of missing values in the data.. This is because not every chemical is tested for in every sample. We will drop columns that have more than 30% of values missing in the training set. These same columns will also be dropped from the test set.

In [11]:
# gets a list of columns
X_train_columns = X_train.columns

# gets a list of missing proportions for each column
missing_props = []
for column in X_train_columns:
  missing_prop = X_train[column].isna().sum() / X_train.shape[0]
  missing_props.append(missing_prop)

# creates a dataframe with the proportion of missing values for each column
train_missing = pd.DataFrame({'column': X_train_columns, 'missing_prop': missing_props})
train_missing_sorted = train_missing.sort_values(by = "missing_prop")
train_missing_sorted.head(30)

,column,missing_prop
187,pH,0.079088
174,Turbidity,0.089129
166,Total Phosphorus (TP),0.098426
147,Specific Conductance (Field),0.111318
180,Water Temperature (Field),0.116400
167,Total Suspended Solids (TSS),0.136110
117,Nitrate (NO3-N),0.142308
15,Ammonia (NH3-N),0.144911
173,True Color,0.151233
165,Total Organic Carbon (TOC),0.176894


In [12]:
# filters for columns with less than 30% of values missing
train_missing_filtered = train_missing[train_missing["missing_prop"] <= 0.3]
train_missing_filtered = train_missing_filtered.sort_values(by = "missing_prop")
train_missing_filtered

,column,missing_prop
187,pH,0.079088
174,Turbidity,0.089129
166,Total Phosphorus (TP),0.098426
147,Specific Conductance (Field),0.111318
180,Water Temperature (Field),0.116400
167,Total Suspended Solids (TSS),0.136110
117,Nitrate (NO3-N),0.142308
15,Ammonia (NH3-N),0.144911
173,True Color,0.151233
165,Total Organic Carbon (TOC),0.176894


In [13]:
# gets a list of columns to keep in the dataframe
columns_to_keep = train_missing_filtered["column"].to_list()
columns_to_keep

['pH',
 'Turbidity',
 'Total Phosphorus (TP)',
 'Specific Conductance (Field)',
 'Water Temperature (Field)',
 'Total Suspended Solids (TSS)',
 'Nitrate (NO3-N)',
 'Ammonia (NH3-N)',
 'True Color',
 'Total Organic Carbon (TOC)',
 'Total Dissolved Phosphorus (TDP)',
 'NOx (Calculated)',
 'Nitrite (NO2-N)',
 'Sulphate (SO4)',
 'Total Alkalinity',
 'Chloride (Cl)',
 'Potassium (K)(Dissolved)',
 'Sodium (Na)(Dissolved)',
 'Magnesium (Mg)(Dissolved)',
 'Calcium (Ca)(Dissolved)',
 'Sodium Adsorption Ratio (SAR)(Calculated)',
 'Dissolved Oxygen (Field)',
 'Total Kjeldahl Nitrogen (TKN)',
 'Hardness',
 'Fluoride (F)']

In [14]:
# creates a new training and testing dataframe with only columns with less than 30% of values missing
X_train_filtered = X_train[columns_to_keep]
X_test_filtered = X_test[columns_to_keep]

In [15]:
# checks for number of columns with less than 30% of values missing
X_train_filtered.shape

(8067, 25)

In [16]:
# defines an iterative imputer with a random forest estimator
imputer = IterativeImputer(estimator = RandomForestRegressor(n_jobs = -1), max_iter=10, random_state=42)
# fits the imputer to the train set and imputes missing values in the training set
X_train_imputed = imputer.fit_transform(X_train_filtered)
# imputes missing values for the test set
X_test_imputed = imputer.transform(X_test_filtered)

c:\Users\kkohl\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [17]:
# creates dataframes for the imputed train and test sets
X_train_imputed_df = pd.DataFrame(X_train_imputed, columns = X_train_filtered.columns, index=X_train_filtered.index)
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns = X_test_filtered.columns, index=X_test_filtered.index)

In [18]:
# concats X_train and y_train
train_set = pd.concat([X_train_imputed_df, y_train], axis = 1)
# concatx X_test and y_test
test_set = pd.concat([X_test_imputed_df, y_test], axis = 1)

In [19]:
# ensures that the indices of the imputed train set and the original train set are the same
train_set.index.equals(X_train_filtered.index)

True

In [20]:
# ensures that the indices of the imputed test set and the original test set are the same
test_set.index.equals(X_test_filtered.index)

True

In [21]:
# ensuring there are no missing values in the imputed train set
train_set.isna().sum()

pH                                           0
Turbidity                                    0
Total Phosphorus (TP)                        0
Specific Conductance (Field)                 0
Water Temperature (Field)                    0
Total Suspended Solids (TSS)                 0
Nitrate (NO3-N)                              0
Ammonia (NH3-N)                              0
True Color                                   0
Total Organic Carbon (TOC)                   0
Total Dissolved Phosphorus (TDP)             0
NOx (Calculated)                             0
Nitrite (NO2-N)                              0
Sulphate (SO4)                               0
Total Alkalinity                             0
Chloride (Cl)                                0
Potassium (K)(Dissolved)                     0
Sodium (Na)(Dissolved)                       0
Magnesium (Mg)(Dissolved)                    0
Calcium (Ca)(Dissolved)                      0
Sodium Adsorption Ratio (SAR)(Calculated)    0
Dissolved Oxy

In [22]:
# ensuring there are no missing values in the imputed test set
test_set.isna().sum()

pH                                           0
Turbidity                                    0
Total Phosphorus (TP)                        0
Specific Conductance (Field)                 0
Water Temperature (Field)                    0
Total Suspended Solids (TSS)                 0
Nitrate (NO3-N)                              0
Ammonia (NH3-N)                              0
True Color                                   0
Total Organic Carbon (TOC)                   0
Total Dissolved Phosphorus (TDP)             0
NOx (Calculated)                             0
Nitrite (NO2-N)                              0
Sulphate (SO4)                               0
Total Alkalinity                             0
Chloride (Cl)                                0
Potassium (K)(Dissolved)                     0
Sodium (Na)(Dissolved)                       0
Magnesium (Mg)(Dissolved)                    0
Calcium (Ca)(Dissolved)                      0
Sodium Adsorption Ratio (SAR)(Calculated)    0
Dissolved Oxy

In [23]:
# exports the train and test sets to csv
train_set.to_csv("train_set.csv", index = False)
test_set.to_csv("test_set.csv", index = False)